# DQN - CartPole Reinforcement Learning

This project implements a Deep Q-Network (DQN) in PyTorch to learn how to balance a pole on a moving cart in the CartPole environment.

Unlike supervised learning, the agent is not given target labels. It learns through interaction with the environment by observing states, choosing actions, receiving rewards, and updating its estimate of the long-term value of each action.

The notebook covers the reinforcement-learning environment, Q-network architecture, experience replay, target networks, epsilon-greedy exploration, DQN optimization, training progress, evaluation, and learned action values.

# 1. Import Required Libraries

Load the libraries needed for reinforcement learning, neural-network training, and visualization.

In [ ]:
!pip install -q "gymnasium[classic-control]"

import random
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym

import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch:", torch.__version__)
print("Gymnasium:", gym.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# 2. Configure Reproducibility and Training

Set the random seed, device, and DQN training parameters.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENV_NAME = "CartPole-v1"

NUM_EPISODES = 500
MAX_STEPS = 500

GAMMA = 0.99
LEARNING_RATE = 1e-3

BATCH_SIZE = 64
REPLAY_CAPACITY = 100000
MIN_REPLAY_SIZE = 1000

EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 30000

TARGET_UPDATE_FREQUENCY = 10

print("Device:", DEVICE)


# 3. Create the CartPole Environment

Create the environment and inspect its state and action spaces.

In [ ]:
env = gym.make(ENV_NAME)

state, info = env.reset(seed=SEED)
env.action_space.seed(SEED)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print("Environment:", ENV_NAME)
print("State dimension:", state_dim)
print("Number of actions:", action_dim)
print("Initial state:", state)

env.close()


# 4. Understand the Reinforcement Learning Setup

Inspect what the agent observes, which actions it can take, and how rewards are assigned.

In [ ]:
env = gym.make(ENV_NAME)

print("State space:", env.observation_space)
print("Action space:", env.action_space)
print("Observation lower bounds:", env.observation_space.low)
print("Observation upper bounds:", env.observation_space.high)

print()
print("Action 0: Move cart left")
print("Action 1: Move cart right")
print()
print("The agent receives a reward of 1 for each time step that the pole remains balanced.")

env.close()


# 5. Build the DQN

Define a neural network that predicts the Q-value of each available action for a given state.

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, state):
        return self.network(state)


policy_net = DQN(state_dim, action_dim).to(DEVICE)
target_net = DQN(state_dim, action_dim).to(DEVICE)

target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

print(policy_net)


# 6. Inspect the Network Parameters

Check the size of the DQN before training.

In [ ]:
total_params = sum(
    parameter.numel()
    for parameter in policy_net.parameters()
)

print("Total parameters:", total_params)


# 7. Create the Experience Replay Buffer

Store previous transitions so the agent can learn from a diverse batch of past experiences.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, terminated, truncated):
        self.buffer.append(
            (state, action, reward, next_state, terminated, truncated)
        )

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)

        states, actions, rewards, next_states, terminated, truncated = zip(*batch)

        return (
            np.asarray(states, dtype=np.float32),
            np.asarray(actions, dtype=np.int64),
            np.asarray(rewards, dtype=np.float32),
            np.asarray(next_states, dtype=np.float32),
            np.asarray(terminated, dtype=np.float32),
            np.asarray(truncated, dtype=np.float32)
        )

    def __len__(self):
        return len(self.buffer)


replay_buffer = ReplayBuffer(REPLAY_CAPACITY)

print("Replay buffer capacity:", REPLAY_CAPACITY)


# 8. Define Exploration and Action Selection

Use epsilon-greedy exploration so the agent gradually shifts from random actions to actions selected by the learned Q-values.

In [ ]:
def get_epsilon(step):
    return EPSILON_END + (
        EPSILON_START - EPSILON_END
    ) * np.exp(-step / EPSILON_DECAY)


def select_action(state, epsilon):
    if random.random() < epsilon:
        return random.randrange(action_dim)

    state_tensor = torch.tensor(
        state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    with torch.no_grad():
        q_values = policy_net(state_tensor)

    return int(q_values.argmax(dim=1).item())


# 9. Define the DQN Optimization Step

Update the policy network using current Q-values and Bellman targets from the target network.

In [ ]:
optimizer = optim.Adam(
    policy_net.parameters(),
    lr=LEARNING_RATE
)

loss_function = nn.SmoothL1Loss()


def optimize_model():
    if len(replay_buffer) < MIN_REPLAY_SIZE:
        return None

    states, actions, rewards, next_states, terminated, truncated = replay_buffer.sample(
        BATCH_SIZE
    )

    states = torch.tensor(states, dtype=torch.float32, device=DEVICE)
    actions = torch.tensor(actions, dtype=torch.long, device=DEVICE).unsqueeze(1)
    rewards = torch.tensor(rewards, dtype=torch.float32, device=DEVICE).unsqueeze(1)
    next_states = torch.tensor(next_states, dtype=torch.float32, device=DEVICE)
    terminated = torch.tensor(terminated, dtype=torch.float32, device=DEVICE).unsqueeze(1)
    truncated = torch.tensor(truncated, dtype=torch.float32, device=DEVICE).unsqueeze(1)

    current_q_values = policy_net(states).gather(1, actions)

    with torch.no_grad():
        next_q_values = target_net(next_states).max(dim=1, keepdim=True).values
        done = torch.maximum(terminated, truncated)
        target_q_values = rewards + GAMMA * next_q_values * (1.0 - done)

    loss = loss_function(current_q_values, target_q_values)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 10.0)
    optimizer.step()

    return float(loss.item())


# 10. Train the DQN Agent

Train the agent through repeated interaction with CartPole while tracking reward, loss, and exploration.

In [ ]:
env = gym.make(ENV_NAME)

episode_rewards = []
episode_losses = []
episode_epsilons = []

global_step = 0
best_rolling_reward = -np.inf

for episode in range(1, NUM_EPISODES + 1):
    state, info = env.reset(seed=SEED + episode)

    total_reward = 0.0
    losses = []

    for step in range(MAX_STEPS):
        epsilon = get_epsilon(global_step)
        action = select_action(state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)

        replay_buffer.add(
            state,
            action,
            reward,
            next_state,
            terminated,
            truncated
        )

        state = next_state
        total_reward += reward
        global_step += 1

        loss = optimize_model()

        if loss is not None:
            losses.append(loss)

        if terminated or truncated:
            break

    if episode % TARGET_UPDATE_FREQUENCY == 0:
        target_net.load_state_dict(policy_net.state_dict())

    episode_rewards.append(total_reward)
    episode_losses.append(np.mean(losses) if losses else np.nan)
    episode_epsilons.append(epsilon)

    rolling_reward = np.mean(episode_rewards[-100:])

    if rolling_reward > best_rolling_reward:
        best_rolling_reward = rolling_reward

    if episode == 1 or episode % 10 == 0:
        print(
            f"Episode {episode:03d} | "
            f"Reward: {total_reward:6.1f} | "
            f"100-Episode Avg: {rolling_reward:6.1f} | "
            f"Epsilon: {epsilon:.3f}"
        )

    if len(episode_rewards) >= 100 and rolling_reward >= 475:
        print(f"Environment solved at episode {episode}.")
        break

env.close()

print("Episodes completed:", len(episode_rewards))
print("Best 100-episode average:", best_rolling_reward)


# 11. Plot Training Progress

Visualize reward progression, moving-average reward, training loss, and exploration decay.

In [ ]:
rewards = np.asarray(episode_rewards, dtype=np.float32)
losses = np.asarray(episode_losses, dtype=np.float32)
epsilons = np.asarray(episode_epsilons, dtype=np.float32)

rolling_window = min(100, len(rewards))
rolling_rewards = pd.Series(rewards).rolling(rolling_window).mean()

plt.figure(figsize=(9, 5))
plt.plot(rewards, label="Episode Reward")
plt.plot(rolling_rewards, label=f"{rolling_window}-Episode Moving Average")
plt.axhline(475, linestyle="--", label="Solved Threshold")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("DQN Training Reward")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(losses)
plt.xlabel("Episode")
plt.ylabel("Average Loss")
plt.title("DQN Training Loss")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(epsilons)
plt.xlabel("Episode")
plt.ylabel("Epsilon")
plt.title("Epsilon-Greedy Exploration Decay")
plt.tight_layout()
plt.show()


# 12. Evaluate the Trained Agent

Run the learned policy without exploration across multiple episodes to measure its final performance.

In [ ]:
eval_env = gym.make(ENV_NAME)

evaluation_rewards = []

for episode in range(20):
    state, info = eval_env.reset(seed=1000 + episode)
    total_reward = 0.0

    for step in range(MAX_STEPS):
        action = select_action(state, epsilon=0.0)

        state, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    evaluation_rewards.append(total_reward)

eval_env.close()

print("Evaluation episodes:", len(evaluation_rewards))
print("Mean evaluation reward:", np.mean(evaluation_rewards))
print("Standard deviation:", np.std(evaluation_rewards))
print("Best evaluation reward:", np.max(evaluation_rewards))
print("Minimum evaluation reward:", np.min(evaluation_rewards))


# 13. Compare Evaluation Rewards

Summarize the agent's performance across the evaluation episodes.

In [ ]:
evaluation_summary = pd.DataFrame({
    "Episode": np.arange(1, len(evaluation_rewards) + 1),
    "Reward": evaluation_rewards
})

display(evaluation_summary)

plt.figure(figsize=(9, 5))
plt.bar(
    evaluation_summary["Episode"],
    evaluation_summary["Reward"]
)
plt.axhline(
    475,
    linestyle="--",
    label="Solved Threshold"
)
plt.xlabel("Evaluation Episode")
plt.ylabel("Reward")
plt.title("DQN Evaluation Rewards")
plt.legend()
plt.tight_layout()
plt.show()


# 14. Inspect Learned Q-Values

Examine the action values produced by the trained network for representative states.

In [ ]:
inspection_env = gym.make(ENV_NAME)

sample_states = []

for seed in [2000, 2001, 2002, 2003, 2004]:
    state, info = inspection_env.reset(seed=seed)
    sample_states.append(state)

inspection_env.close()

sample_tensor = torch.tensor(
    np.asarray(sample_states),
    dtype=torch.float32,
    device=DEVICE
)

with torch.no_grad():
    q_values = policy_net(sample_tensor).cpu().numpy()

q_values_df = pd.DataFrame(
    q_values,
    columns=["Q(Action 0)", "Q(Action 1)"]
)

display(q_values_df)


# 15. Key Findings



# 16. Conclusion

